# FSRE-AdaTSK: reproducing the article step by step

This notebook reproduces eight rows of Table IV of

> G. Xue, Q. Chang, J. Wang, K. Zhang and N. R. Pal, "An adaptive neuro-fuzzy system with
> integrated feature selection and rule extraction for high-dimensional classification
> problems", *IEEE Transactions on Fuzzy Systems*, vol. 31, no. 7, pp. 2167-2181, 2023.

It runs the same code as the script `examples/reproductions/fsre_adatsk_2022.py`, one
step at a time:

1. settings and reproducibility;
2. the data;
3. one fold in detail: the three phases of the method;
4. the full protocol of the article, ten-fold cross-validation repeated five times;
5. the comparison with the article.

The full protocol takes about an hour and a half on a laptop.

## 1. Settings and reproducibility

In [1]:
import warnings

import numpy as np
import torch
from sklearn.datasets import fetch_openml, load_breast_cancer, load_iris, load_wine
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

import highfis
from highfis import FSREADATSKClassifier

The seed, the numbers of repetitions and folds, and the number of threads are fixed here.
The values of the article are copied from its Table IV.

In [2]:
SEED = 0
REPETITIONS = 5
FOLDS = 10
THREADS = 1

# Datasets that ship with scikit-learn, and OpenML identifiers of the others.
BUILT_IN = {"Iris": load_iris, "Wine": load_wine, "Wdbc": load_breast_cancer}
OPENML = {"Appendicitis": 1456, "Pima": 37, "Glass": 41, "Heart": 53, "Spectfheart": 1600}
DATASETS = ["Iris", "Appendicitis", "Pima", "Glass", "Wine", "Heart", "Wdbc", "Spectfheart"]

# Table IV of the article: accuracy in percent, selected features, extracted rules.
ARTICLE = {
    "Iris": (96.5, 2.1, 6.1),
    "Appendicitis": (86.3, 3.7, 3.1),
    "Pima": (75.7, 3.0, 3.2),
    "Glass": (65.5, 4.1, 6.0),
    "Wine": (97.3, 6.3, 6.3),
    "Heart": (82.5, 7.4, 2.9),
    "Wdbc": (95.4, 6.2, 4.9),
    "Spectfheart": (79.2, 3.4, 5.1),
}

Three lines make the run repeatable: a fixed number of threads, deterministic algorithms
in PyTorch, and no reuse of fuzzy sets between fits.

In [3]:
torch.set_num_threads(THREADS)
torch.use_deterministic_algorithms(True)
highfis.set_mf_cache_enabled(False)

Glass has a class with nine samples, fewer than the ten folds, and scikit-learn warns
about it at every split. The warning is silenced here.

In [4]:
warnings.filterwarnings("ignore", message="The least populated class")

## 2. The data

Three datasets ship with scikit-learn and five are downloaded from OpenML on the first
run. On OpenML the class of Appendicitis is the last column of the data.

In [5]:
def load(name: str) -> tuple[np.ndarray, np.ndarray]:
    """Return the features and the labels, numbered from zero, of a dataset."""
    if name in BUILT_IN:
        features, labels = BUILT_IN[name](return_X_y=True)
    else:
        bunch = fetch_openml(data_id=OPENML[name], as_frame=False, parser="liac-arff")
        features, labels = bunch.data, bunch.target
        if name == "Appendicitis":  # on OpenML the class is the last column of the data
            features, labels = features[:, :-1], features[:, -1]
    return features.astype(np.float32), np.unique(labels, return_inverse=True)[1]


for name in DATASETS:
    features, labels = load(name)
    print(f"{name}: {features.shape[0]} samples, {features.shape[1]} features, {labels.max() + 1} classes")

Iris: 150 samples, 4 features, 3 classes
Appendicitis: 106 samples, 7 features, 2 classes
Pima: 768 samples, 8 features, 2 classes
Glass: 214 samples, 9 features, 6 classes
Wine: 178 samples, 13 features, 3 classes
Heart: 270 samples, 13 features, 2 classes
Wdbc: 569 samples, 30 features, 2 classes
Spectfheart: 267 samples, 44 features, 2 classes


## 3. One fold in detail

Before running the whole protocol, this section follows the first fold of Wine.

### The split and the scaling

The article does not state how the inputs are scaled. Here they are standardized, with
the mean and the standard deviation of the training part. The choice matters: the
membership function of the article, $e^{-(x - m)^2}$, has no spread, so the scale of a
feature decides how far apart its fuzzy sets are.

In [6]:
features, labels = load("Wine")

folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
train, test = next(folds.split(features, labels))

scaler = StandardScaler().fit(features[train])
x_train, x_test = scaler.transform(features[train]), scaler.transform(features[test])

print(f"training part: {x_train.shape[0]} samples, test part: {x_test.shape[0]} samples")

training part: 160 samples, test part: 18 samples


### The three phases

FSRE-AdaTSK trains in three phases:

1. **Feature selection.** Ten fuzzy sets per feature and ten rules; one gate per feature
   is trained with the system, and the features whose gates open are kept.
2. **Rule extraction.** Five fuzzy sets on each selected feature, and an enhanced rule
   base with $(2D' + 1) \times 5$ rules for $D'$ selected features; one gate per rule is
   trained, and the rules whose gates open are kept.
3. **Fine-tuning** of what remains, without gates.

The arguments below are the settings of the article: the numbers of fuzzy sets of the
two first phases and no normalization of the consequent inputs. The article does not
state the number of iterations; the rule-extraction phase runs for 600 here.

In [7]:
model = FSREADATSKClassifier(
    fs_n_mfs=10,
    re_n_mfs=5,
    re_epochs=600,
    consequent_batch_norm=False,
    random_state=SEED,
).fit(x_train, labels[train])

n_selected = len(model.selected_features_)
print(f"selected features: {n_selected} of {x_train.shape[1]}")
print(f"candidate rules in the rule-extraction phase: {(2 * n_selected + 1) * 5}")
print(f"extracted rules: {model.model_.n_rules}")
print(f"test accuracy: {100.0 * model.score(x_test, labels[test]):.2f}%")

selected features: 9 of 13
candidate rules in the rule-extraction phase: 95
extracted rules: 5
test accuracy: 100.00%


The selected features, by their names in the dataset:

In [8]:
feature_names = load_wine().feature_names
print([feature_names[d] for d in model.selected_features_])

['alcohol', 'ash', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']


## 4. The full protocol

The article reports the mean over ten-fold cross-validation repeated five times, that is,
50 runs per dataset. The loop below repeats the previous section for every run and stores
the accuracy and the numbers of selected features and extracted rules.

In [9]:
results = {name: [] for name in DATASETS}

for name in DATASETS:
    features, labels = load(name)

    for repetition in range(REPETITIONS):
        folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED + repetition)
        for train, test in folds.split(features, labels):
            scaler = StandardScaler().fit(features[train])
            x_train, x_test = scaler.transform(features[train]), scaler.transform(features[test])

            model = FSREADATSKClassifier(
                fs_n_mfs=10,
                re_n_mfs=5,
                re_epochs=600,
                consequent_batch_norm=False,
                random_state=SEED + repetition,
            ).fit(x_train, labels[train])
            results[name].append(
                (100.0 * model.score(x_test, labels[test]), len(model.selected_features_), model.model_.n_rules)
            )
    print(f"{name}: {len(results[name])} runs done")

Iris: 50 runs done


Appendicitis: 50 runs done


Pima: 50 runs done


Glass: 50 runs done


Wine: 50 runs done


Heart: 50 runs done


Wdbc: 50 runs done


Spectfheart: 50 runs done


## 5. Comparison with the article

In [10]:
print("Table IV: FSRE-AdaTSK (accuracy in percent / selected features / extracted rules)")
print("std of accuracy: over the runs, and over the means of the repetitions")
print(f"{'dataset':12s} {'article':>18s} {'highFIS':>20s} {'std of accuracy':>16s}")
for name in DATASETS:
    accuracy, n_features, n_rules = np.asarray(results[name]).T
    reported = "{:.1f} / {:.1f} / {:.1f}".format(*ARTICLE[name])
    measured = f"{accuracy.mean():.2f} / {n_features.mean():.2f} / {n_rules.mean():.2f}"
    deviations = f"{accuracy.std():.2f} / {accuracy.reshape(REPETITIONS, FOLDS).mean(axis=1).std():.2f}"
    print(f"{name:12s} {reported:>18s} {measured:>20s} {deviations:>16s}")

Table IV: FSRE-AdaTSK (accuracy in percent / selected features / extracted rules)
std of accuracy: over the runs, and over the means of the repetitions
dataset                 article              highFIS  std of accuracy
Iris           96.5 / 2.1 / 6.1  96.40 / 2.00 / 6.20      4.46 / 0.33
Appendicitis   86.3 / 3.7 / 3.1  84.96 / 3.14 / 3.24      8.40 / 1.34
Pima           75.7 / 3.0 / 3.2  76.30 / 3.40 / 3.28      4.45 / 0.30
Glass          65.5 / 4.1 / 6.0  65.35 / 5.66 / 6.00      9.71 / 0.92
Wine           97.3 / 6.3 / 6.3  96.18 / 8.22 / 6.80      5.80 / 1.11
Heart          82.5 / 7.4 / 2.9  82.07 / 8.88 / 2.96      6.84 / 2.51
Wdbc           95.4 / 6.2 / 4.9  95.26 / 7.12 / 4.36      2.61 / 0.37
Spectfheart    79.2 / 3.4 / 5.1  77.17 / 5.40 / 5.70      6.81 / 1.80


The accuracies are within two points of the article on the eight datasets, and the
numbers of extracted rules are reproduced. The number of selected features is the
article's on Iris, Appendicitis and Pima, and one or two above it on the others.

The standard deviations of the article, 0.4 to 1.5 points on these datasets, have the
size of a deviation between the means of the five repetitions. The means of each
repetition here:

In [11]:
for name in DATASETS:
    accuracy = np.asarray(results[name])[:, 0].reshape(REPETITIONS, FOLDS)
    print(f"{name:12s} mean of each repetition: {np.round(accuracy.mean(axis=1), 2).tolist()}  std: {accuracy.mean(axis=1).std():.2f}")

Iris         mean of each repetition: [96.67, 96.67, 96.0, 96.0, 96.67]  std: 0.33
Appendicitis mean of each repetition: [86.0, 83.0, 84.0, 86.73, 85.09]  std: 1.34
Pima         mean of each repetition: [76.3, 76.43, 76.69, 75.77, 76.3]  std: 0.30
Glass        mean of each repetition: [64.46, 65.95, 64.52, 66.86, 64.98]  std: 0.92
Wine         mean of each repetition: [95.56, 95.56, 97.78, 97.19, 94.84]  std: 1.11
Heart        mean of each repetition: [81.85, 82.96, 85.93, 78.15, 81.48]  std: 2.51
Wdbc         mean of each repetition: [94.9, 95.79, 95.07, 94.91, 95.61]  std: 0.37
Spectfheart  mean of each repetition: [78.99, 78.69, 77.91, 74.2, 76.08]  std: 1.80


## Environment

The versions that produced the numbers above:

In [12]:
highfis.show_versions()

highfis       : 0.33.0
python        : 3.11.15
platform      : Linux-7.0.0-38-generic-x86_64-with-glibc2.43
torch         : 2.14.1+cu130
numpy         : 2.4.6
scikit-learn  : 1.9.1
default dtype : torch.float32
threads       : 1
cpu           : Intel(R) Core(TM) Ultra 7 258V
